<a href="https://colab.research.google.com/github/prasad2154/NewVision/blob/main/GenAI/RAG/11_production_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production RAG Pipeline

We've learned the pieces. Today, we put them together.

**PDF → Chunking → Vector DB → Retrieval → LLM** — a complete RAG system.

## Install PDF Library

In [1]:
%pip install PyPDF2 -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

## Setup

In [2]:
from google import genai
import chromadb
from PyPDF2 import PdfReader
import os
from dotenv import load_dotenv

from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')
# load_dotenv(dotenv_path='./.env')
# API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=API_KEY)

## Step 1: Load PDF

In [7]:
# Load PDF
pdf_path = "/content/leave_policy.pdf"  # Your PDF file
reader = PdfReader(pdf_path)

# Extract text from all pages
full_text = ""   # empty string
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"📄 Loaded PDF: {len(reader.pages)} pages, {len(full_text)} characters")
print(f"\n📖 Preview:\n{full_text[:500]}...")

📄 Loaded PDF: 8 pages, 19644 characters

📖 Preview:
 
 Pride Global India Employee Leave Policy  
PRIDE Technologies Consulting (India) Private Limited (Ver. 2.0) 
Russell Tobin Associates Staffing Solutions India Private Limited (Ver. 2.0) 
 
Date  Version  Rational change  Description  
15 January, 2025  1.0 Annual review  Periodic review of leave 
provisions and 
procedures to align 
with company and end 
client requirements  
6 March 2026  2.0 Annual review  Periodic review of leave 
provisions and 
procedures to align 
with company and end 
...


## Step 2: Chunk the Document

In [4]:
# divide all data characters chucks divide into 500 each
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()  # fixed chunking: 500 chars.
        if chunk:
            chunks.append(chunk)
        start = end - overlap   # 50 steps back  0-500  500-50 = 450
    return chunks

chunks = chunk_text(full_text)

print(f"✂️ Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}: {chunk[:80]}...\n")

✂️ Created 44 chunks

Chunk 0: Pride Global India Employee Leave Policy  
PRIDE Technologies Consulting (India)...

Chunk 1: s and 
procedures to align 
with company and end 
client requirements  
 
 
  
V...

Chunk 2: ...........  3 
Public Holidays  ................................ .................



## Step 3: Store in Vector Database

In [5]:
# Create ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_store2")

# Generate embeddings and store
for i, chunk in enumerate(chunks):
    embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    ).embeddings[0].values

   # add chunks and embeddings in the chroma collection
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[f"chunk_{i}"]
    )

print(f"✅ Stored {collection.count()} chunks in ChromaDB")

✅ Stored 44 chunks in ChromaDB


## Step 4: The RAG Function

In [16]:
def ask(question):
    """Complete RAG pipeline: retrieve + generate."""

    # Embed the question
    query_emb = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    ).embeddings[0].values

    # Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3
    )
    context = "\n\n".join(results['documents'][0])

    # Generate answer
    prompt = f"""Answer the question based ONLY on the context below.
If the answer is not in the context, say "I couldn't find that information."

Context:
{context}

Question: {question}

Answer:"""

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text, results['documents'][0]

## Test: Ask Questions About the PDF

In [17]:
question = "What is sick leave?"

answer, sources = ask(question)

print(f"❓ {question}\n")
print(f"💬 {answer}\n")
print("📄 Sources:")
for i, src in enumerate(sources[1:]):
    print(f"  [{i+1}] {src[:100]}...")

❓ What is sick leave?

💬 Sick leave is time-off allotted on a pro-rata basis to address health and wellness concerns.

📄 Sources:
  [1] ate for resuming duty after sickness, from a registered medical practitioner.  
Leave application (f...
  [2] be allowed to resume work only after 
the prescribed rest period. In order to resume his/her regular...


In [18]:
# Try your own questions!
question = input("Ask a question: ")
answer, sources = ask(question)
print(f"\n💬 {answer}")

Ask a question: can u give me details of Privilage leaves at pune location

💬 At the Pune location, the Annual S&E Entitlement for Privilege leaves is 18 days, and the maximum accumulation allowed is 45 days.


In [19]:
# Try your own questions!
question = input("Ask a question: ")
answer, sources = ask(question)
print(f"\n💬 {answer}")

Ask a question: how many total leaves?

💬 I couldn't find that information.


## The Complete Pipeline

```
PDF → Extract Text → Chunk → Embed → Store in Vector DB
                                        ↓
Question → Embed → Search → Retrieve → LLM → Answer
```

## Key Takeaways

1. **Extract** text from PDFs using PyPDF2
2. **Chunk** with overlap to preserve context
3. **Store** embeddings in a vector database
4. **Retrieve** and **Generate** — that's RAG

---

This wraps up RAG basics. **Next section: Agents** — when retrieval isn't enough and you need to take actions.

# **What is Hallucination**
# **How to reduce or handle the hallucination**
# **If ur model is not giving you the expected answer then how to handle this problem??**

# **If we have pdf with text, image, table, image of table then how to embed this data/handle this case???**

# **If we have pdfs cvs excel image files in GB's then how to handle the embedding of this????**